In [0]:
"""

===============================================================================
Procedure: Load Silver Table for Product information (Bronze -> Silver)
===============================================================================
Script Purpose:
    This stored procedure performs the ETL (Extract, Transform, Load) process to 
    populate the 'silver' schema tables from the 'bronze' schema.
	Actions Performed:
		- Truncates Silver tables if already there.
		- Inserts transformed and cleansed data from Bronze into Silver tables.
	Objectives:
		1-Extract the prd catagory 
		2-Extract product key
		3-Handle NUll Cost and Revenue columns
		4-Map Product line codes to the descriptive names 

"""

In [0]:
#init 

catalog_name = "abhi_dwh_sql_based"
source_schema  = "bronze"
sink_schema = "silver"
table_name = "crm_prd_info"


# Read the cust_info from the bronze layer 

In [0]:
df = spark.read.table(f"{catalog_name}.{source_schema}.{table_name}")



In [0]:
df.show(5)

## Transformation to clean the data

In [0]:
#import necessory library 

from pyspark.sql.functions import col, row_number, replace, regexp_replace, substring, length, coalesce, lit, upper, when, trim
from pyspark.sql.window import Window 

### 0. Trim all the leading and trailing space from all the columns 

In [0]:
df = df.select(*[trim(col(c)).alias(c) if dict(df.dtypes)[c] == 'string' else col(c) for c in df.columns])

### 1.Separate the product catagory and product key from prd_key

In [0]:
df = df.withColumn("prd_cat", substring(col("prd_key"),1,5)).withColumn("prd_key", substring(col("prd_key"),7,length(col("prd_key"))))
df.show(5)

### 2.Handle NUll values 

In [0]:
#drops row with null value in the prd_id 
df = df.dropna(subset=["prd_id"])
df.show(5)

In [0]:
# replace null in the prd_cost column with 0 
df = df.withColumn("prd_cost", coalesce(col("prd_cost"), lit(0)))
df.show(5)

### 3.Map the prd_line codes with the descriptive names 


In [0]:
df.select(col("prd_line")).distinct().show()

In [0]:
prd_line_code_map = {"M":"Mountain","R":"Road","T":"Touring","S":"Specialized"}
expr = None
for k,v in prd_line_code_map.items():
    expr = when(upper(col("prd_line")) == k, v) if expr is None else expr.when(upper(col("prd_line")) == k , v)   


df = df.withColumn("prd_line", expr.otherwise("n/a"))
df.show(5)

In [0]:
df.select('prd_line').distinct().show()

# Write it to Silver Layer after cleansing 

In [0]:
df.write.mode("overwrite").saveAsTable(f"{catalog_name}.{sink_schema}.{table_name}")


In [0]:
df.display(5)